# 01 Credit Risk Scorecard & ML Pipeline
**Portfolio Project 2: Consumer Credit Default Risk**
- Weight of Evidence (WOE) & Information Value (IV)
- Logistic Regression Baseline Scorecard (PDO Scaling: 300-850)
- Challenger XGBoost Classifier
- Out-of-Time (OOT) Train/Test Validation
- SHAP Feature Importance & Regulatory Explainability

In [1]:
"""
01_credit_risk_scorecard.py - Production Credit Risk Scorecard & ML Pipeline

Performs:
1. Data loading and Out-of-Time (OOT) train/test validation
2. Continuous feature fine-binning, Weight of Evidence (WOE) & Information Value (IV) calculation
3. Feature selection (0.02 <= IV <= 0.50) and multicollinearity inspection
4. Baseline Scorecard: Logistic Regression trained on WOE features
5. Challenger Classifier: XGBoost model with depth restriction and probability calibration
6. Rigorous evaluation: ROC-AUC, Kolmogorov-Smirnov (KS) statistic, Gini, PR-AUC, Brier score
7. Industry standard Points-to-Double-Odds (PDO) scaling to 300-850 FICO-style credit score
8. SHAP TreeExplainer feature attribution for regulatory adverse action explainability
9. Export of scored evaluation datasets for Excel simulator and GenAI modules
"""

import os
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    auc, brier_score_loss, classification_report
)
import xgboost as xgb
import shap


# -----------------------------------------------------------------------------
# 1. Weight of Evidence (WOE) & Information Value (IV) Engine
# -----------------------------------------------------------------------------
class WOEEngine:
    def __init__(self, n_bins=6):
        self.n_bins = n_bins
        self.binning_maps = {}
        self.iv_summary = {}

    def fit(self, X, y):
        """Fits quantile binning and calculates WOE/IV for continuous and categorical features."""
        for col in X.columns:
            series = X[col]
            is_numeric = np.issubdtype(series.dtype, np.number)
            
            if is_numeric and series.nunique() > self.n_bins:
                try:
                    binned, bin_edges = pd.qcut(series, q=self.n_bins, retbins=True, duplicates='drop')
                except Exception:
                    binned, bin_edges = pd.cut(series, bins=self.n_bins, retbins=True)
            else:
                binned = series.astype(str)
                bin_edges = None

            temp_df = pd.DataFrame({'bin': binned.astype(str), 'target': y})
            grouped = temp_df.groupby('bin', observed=False)['target'].agg(
                total='count',
                bad='sum'
            ).reset_index()
            grouped['good'] = grouped['total'] - grouped['bad']

            total_good = (y == 0).sum()
            total_bad = (y == 1).sum()

            # Laplace smoothing (+0.5) to avoid division by zero
            grouped['dist_good'] = (grouped['good'] + 0.5) / (total_good + 1.0)
            grouped['dist_bad'] = (grouped['bad'] + 0.5) / (total_bad + 1.0)

            grouped['woe'] = np.log(grouped['dist_good'] / grouped['dist_bad'])
            grouped['iv_bin'] = (grouped['dist_good'] - grouped['dist_bad']) * grouped['woe']
            total_iv = grouped['iv_bin'].sum()

            self.binning_maps[col] = {
                'is_numeric': is_numeric,
                'bin_edges': bin_edges,
                'woe_lookup': dict(zip(grouped['bin'], grouped['woe'])),
                'iv': total_iv
            }
            self.iv_summary[col] = total_iv

        return self

    def transform(self, X):
        """Replaces raw feature values with their calculated WOE values."""
        X_woe = pd.DataFrame(index=X.index)
        for col in X.columns:
            col_info = self.binning_maps[col]
            series = X[col]
            woe_map = col_info['woe_lookup']
            
            if col_info['is_numeric'] and col_info['bin_edges'] is not None:
                binned = pd.cut(series, bins=col_info['bin_edges'], include_lowest=True).astype(str)
            else:
                binned = series.astype(str)
                
            default_woe = np.mean(list(woe_map.values()))
            X_woe[f"{col}_woe"] = binned.map(woe_map).fillna(default_woe)
            
        return X_woe


# -----------------------------------------------------------------------------
# 2. Points-to-Double-Odds (PDO) Scorecard Scaler
# -----------------------------------------------------------------------------
class ScorecardScaler:
    """
    Calibrates predicted default probabilities P to 300-850 FICO-style credit score:
    Odds = (1 - P) / P
    Score = Offset + Factor * ln(Odds)
    Configured so:
      P = 0.02 -> ~800 (Super Prime)
      P = 0.05 -> ~740 (Grade A)
      P = 0.12 -> ~670 (Grade B)
      P = 0.18 -> ~640 (Optimal Profit Cut-Off Threshold)
      P = 0.28 -> ~600 (Subprime)
      P = 0.50+ -> <540 (Deep Subprime)
    """
    def __init__(self, base_score=640, base_odds=4.55, pdo=46.0):
        self.base_score = base_score
        self.base_odds = base_odds
        self.pdo = pdo
        self.factor = pdo / np.log(2.0)
        self.offset = base_score - self.factor * np.log(base_odds)

    def prob_to_score(self, probs):
        probs = np.clip(probs, 0.001, 0.999)
        odds = (1.0 - probs) / probs
        scores = self.offset + self.factor * np.log(odds)
        return np.clip(np.round(scores), 300, 850).astype(int)


# -----------------------------------------------------------------------------
# 3. Model Evaluation Suite: KS, AUC, Gini, Brier
# -----------------------------------------------------------------------------
def calculate_ks_statistic(y_true, y_prob):
    """Calculates Kolmogorov-Smirnov (KS) statistic: max |TPR - FPR|."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks_stat = np.max(np.abs(tpr - fpr))
    return ks_stat


def evaluate_model_performance(name, y_true, y_prob):
    roc_auc = roc_auc_score(y_true, y_prob)
    ks_stat = calculate_ks_statistic(y_true, y_prob)
    gini = 2.0 * roc_auc - 1.0
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)
    brier = brier_score_loss(y_true, y_prob)

    print(f"\n{'='*25} {name} Performance {'='*25}")
    print(f"ROC-AUC:           {roc_auc:.4f}  (Target >= 0.72)")
    print(f"KS Statistic:      {ks_stat*100.0:.2f}% (Target 38.0% - 45.0%)")
    print(f"Gini Coefficient:  {gini:.4f}")
    print(f"PR-AUC:            {pr_auc:.4f}")
    print(f"Brier Score:       {brier:.4f}")
    print(f"{'='*65}")

    return {
        'model': name,
        'roc_auc': roc_auc,
        'ks_statistic': ks_stat,
        'gini': gini,
        'pr_auc': pr_auc,
        'brier_score': brier
    }


# -----------------------------------------------------------------------------
# Main Execution Pipeline
# -----------------------------------------------------------------------------
def run_scorecard_pipeline():
    script_dir = os.path.dirname(os.path.abspath(__file__))
    project_dir = os.path.dirname(script_dir)
    data_dir = os.path.join(project_dir, 'data')
    train_path = os.path.join(data_dir, 'train_loans.csv')
    test_path = os.path.join(data_dir, 'test_loans.csv')

    print("=" * 70)
    print("02_Credit_Risk: Scorecard & Predictive Modeling Pipeline")
    print("=" * 70)

    # 1. Load Data
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print(f"Loaded Train set: {train_df.shape[0]} records, default rate: {train_df['default_flag'].mean():.2%}")
    print(f"Loaded Test set:  {test_df.shape[0]} records, default rate: {test_df['default_flag'].mean():.2%}")

    # 2. Candidate Features for Modeling
    continuous_features = [
        'dti', 'annual_inc', 'revol_util', 'loan_amnt',
        'int_rate', 'inq_last_6mths', 'open_acc', 'delinq_2yrs'
    ]
    categorical_features = [
        'home_ownership', 'verification_status', 'purpose'
    ]
    all_candidate_features = continuous_features + categorical_features

    X_train_raw = train_df[all_candidate_features].copy()
    y_train = train_df['default_flag'].values

    X_test_raw = test_df[all_candidate_features].copy()
    y_test = test_df['default_flag'].values

    # 3. Fit WOE Engine
    print("\nCalculating Weight of Evidence (WOE) and Information Value (IV)...")
    woe_engine = WOEEngine(n_bins=6)
    woe_engine.fit(X_train_raw, y_train)

    print("\nInformation Value (IV) Ranking:")
    print(f"{'Feature':<25} {'Information Value':<20} {'Predictive Power'}")
    print("-" * 65)
    selected_features = []
    for feat, iv in sorted(woe_engine.iv_summary.items(), key=lambda x: x[1], reverse=True):
        if iv < 0.02:
            power = "Unpredictive (<0.02) - DROP"
        elif iv < 0.10:
            power = "Weak (0.02 - 0.10) - RETAIN"
            selected_features.append(feat)
        elif iv < 0.30:
            power = "Medium (0.10 - 0.30) - RETAIN"
            selected_features.append(feat)
        elif iv <= 0.50:
            power = "Strong (0.30 - 0.50) - RETAIN"
            selected_features.append(feat)
        else:
            power = "Suspiciously High (>0.50) - CHECK LEAKAGE"
            selected_features.append(feat)
        print(f"{feat:<25} {iv:<20.4f} {power}")

    print(f"\nRetained {len(selected_features)} features satisfying IV rule (0.02 <= IV <= 0.50):")
    print(selected_features)

    # 4. Transform to WOE space for Baseline Logistic Regression
    X_train_selected = X_train_raw[selected_features]
    X_test_selected = X_test_raw[selected_features]

    X_train_woe = woe_engine.transform(X_train_selected)
    X_test_woe = woe_engine.transform(X_test_selected)

    # Train Baseline Logistic Regression
    print("\nTraining Baseline Scorecard (Logistic Regression on WOE)...")
    lr_model = LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42)
    lr_model.fit(X_train_woe, y_train)

    lr_probs_train = lr_model.predict_proba(X_train_woe)[:, 1]
    lr_probs_test = lr_model.predict_proba(X_test_woe)[:, 1]

    lr_perf = evaluate_model_performance("Baseline Logistic Regression (WOE Scorecard)", y_test, lr_probs_test)

    # 5. Train Challenger XGBoost Model
    print("\nTraining Challenger Classifier (XGBoost)...")
    # For XGBoost, encode categoricals
    X_train_xgb = pd.get_dummies(X_train_raw, columns=categorical_features, drop_first=True)
    X_test_xgb = pd.get_dummies(X_test_raw, columns=categorical_features, drop_first=True)
    # Align columns
    X_train_xgb, X_test_xgb = X_train_xgb.align(X_test_xgb, join='left', axis=1, fill_value=0)

    scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
    xgb_model = xgb.XGBClassifier(
        n_estimators=160,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train_xgb, y_train)

    xgb_probs_test = xgb_model.predict_proba(X_test_xgb)[:, 1]
    xgb_perf = evaluate_model_performance("Challenger XGBoost Classifier", y_test, xgb_probs_test)

    # 6. Scorecard Calibration & PDO Scaling
    print("\nApplying Points-to-Double-Odds (PDO) Scorecard Scaling (300 - 850)...")
    # Unweight XGBoost probabilities (compensating for scale_pos_weight)
    weighted_odds = (1.0 - xgb_probs_test) / np.clip(xgb_probs_test, 1e-6, 1.0 - 1e-6)
    true_odds = weighted_odds * scale_pos_weight
    calibrated_probs = 1.0 / (1.0 + true_odds)

    # Scorecard scaling: Factor = 60.0 (PDO ~41.6), Offset = 550.0
    # Yields: P=0.03 -> ~760, P=0.12 -> ~670, P=0.18 -> 640 (Cut-Off), P=0.25 -> 615, P=0.50+ -> <550
    scaler = ScorecardScaler(base_score=640, base_odds=4.55, pdo=41.6)
    credit_scores = scaler.prob_to_score(calibrated_probs)

    test_df['predicted_default_prob'] = np.round(calibrated_probs, 4)
    test_df['calibrated_credit_score'] = credit_scores

    print("Calibrated Credit Score Distribution on Test Set:")
    print(f"  Mean Score:   {credit_scores.mean():.1f}")
    print(f"  Median Score: {np.median(credit_scores):.1f}")
    print(f"  Min Score:    {credit_scores.min()}")
    print(f"  Max Score:    {credit_scores.max()}")

    # 7. SHAP Explainability & Top Adverse Drivers
    print("\nComputing SHAP TreeExplainer Attributions for Model Governance...")
    explainer = shap.TreeExplainer(xgb_model)
    # Take sample for fast, robust explanation
    sample_size = min(1500, len(X_test_xgb))
    X_sample = X_test_xgb.iloc[:sample_size]
    shap_values = explainer.shap_values(X_sample)

    feature_names = list(X_test_xgb.columns)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_global_drivers = pd.DataFrame({
        'feature': feature_names,
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    print("\nTop 5 Global Model Risk Drivers (TreeSHAP):")
    for idx, row in top_global_drivers.head(5).iterrows():
        print(f"  {idx+1}. {row['feature']:<20} (Mean |SHAP| = {row['mean_abs_shap']:.4f})")

    # Extract individual negative drivers for declined applicants
    # High risk cutoff threshold: score < 640
    test_df['application_decision'] = test_df['calibrated_credit_score'].apply(
        lambda s: 'APPROVED' if s >= 640 else 'DECLINED'
    )
    
    # Calculate SHAP values for all test instances
    shap_all = explainer.shap_values(X_test_xgb)
    
    top_driver_1 = []
    top_driver_2 = []
    top_driver_3 = []
    
    for i in range(len(X_test_xgb)):
        row_shap = shap_all[i]
        # Positive SHAP pushes toward default (risk)
        pos_indices = np.argsort(row_shap)[::-1]
        d1 = f"{feature_names[pos_indices[0]]}: {X_test_xgb.iloc[i][feature_names[pos_indices[0]]]:.1f}"
        d2 = f"{feature_names[pos_indices[1]]}: {X_test_xgb.iloc[i][feature_names[pos_indices[1]]]:.1f}"
        d3 = f"{feature_names[pos_indices[2]]}: {X_test_xgb.iloc[i][feature_names[pos_indices[2]]]:.1f}"
        top_driver_1.append(d1)
        top_driver_2.append(d2)
        top_driver_3.append(d3)

    test_df['adverse_reason_1'] = top_driver_1
    test_df['adverse_reason_2'] = top_driver_2
    test_df['adverse_reason_3'] = top_driver_3

    # 8. Export Scored Dataset
    output_scored_path = os.path.join(data_dir, 'scored_test_loans.csv')
    test_df.to_csv(output_scored_path, index=False)
    print(f"\nExported scored test loans to: {output_scored_path} ({len(test_df)} rows)")

    # 9. Verify Approval & Default Trade-Off at Cutoff 640
    approved = test_df[test_df['application_decision'] == 'APPROVED']
    declined = test_df[test_df['application_decision'] == 'DECLINED']

    print("\n" + "=" * 70)
    print("Policy Decision Summary at Score Cut-off = 640:")
    print(f"  Approval Rate:        {len(approved)/len(test_df)*100.0:.2f}% ({len(approved)} approved / {len(test_df)} total)")
    print(f"  Approved Bad Rate:    {approved['default_flag'].mean()*100.0:.2f}% (vs. baseline uncalibrated {test_df['default_flag'].mean()*100.0:.2f}%)")
    print(f"  Declined Bad Rate:    {declined['default_flag'].mean()*100.0:.2f}%")
    print(f"  Relative Risk Drop:   {(test_df['default_flag'].mean() - approved['default_flag'].mean()) / test_df['default_flag'].mean() * 100.0:.1f}% reduction in default rate")
    print("=" * 70)
    print("Credit risk scorecard pipeline completed successfully.")


if __name__ == '__main__':
    run_scorecard_pipeline()
